# Emergency Keyword Flag Pipeline (Execution)


This notebook builds **one binary column per emergency keyword** using only text fields:
- `chief_complaint_text`
- `injury_cause_text`


## Rules implemented
- Strict NLP normalization and abbreviation expansion
- Strict negation handling (`no chest pain` does **not** trigger)
- One column per keyword (`0/1`)
- Output file contains only `row_index` + keyword flag columns


## Output
- `working_data/nhamcs_emergency_keyword_flags_only.csv`
- `working_data/emergency_keyword_catalog.csv`

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_PATH = Path("../working_data/nhamcs_data_2018_22.csv")
FINAL_OUTPUT_PATH = Path("../working_data/nhamcs_emergency_keyword_flags_matched_only.csv")
TEXT_COLS = ["chief_complaint_text", "injury_cause_text"]

df = pd.read_csv(DATA_PATH)
missing_cols = [c for c in TEXT_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required text columns: {missing_cols}")

ABBREV_MAP = {
    "sob": "shortness of breath",
    "cp": "chest pain",
    "loc": "loss of consciousness",
    "ams": "altered mental status",
    "sz": "seizure",
    "mi": "myocardial infarction",
    "cva": "stroke",
    "od": "overdose",
    "si": "suicidal ideation",
    "hi": "homicidal ideation",
}

PLACEHOLDER_VALUES = {
    "", "blank", "unknown", "unknown/blank", "blank unknown", "nan", "none", "na", "n a",
}

EMERGENCY_KEYWORDS = [
    "heart attack",
    "myocardial infarction",
    "cardiac arrest",
    "chest pain",
    "chest pressure",
    "chest tightness",
    "stroke",
    "slurred speech",
    "facial droop",
    "unilateral weakness",
    "one sided weakness",
    "choking",
    "airway obstruction",
    "respiratory distress",
    "shortness of breath",
    "difficulty breathing",
    "cannot breathe",
    "anaphylaxis",
    "severe allergic reaction",
    "throat swelling",
    "tongue swelling",
    "seizure",
    "status epilepticus",
    "altered mental status",
    "unresponsive",
    "loss of consciousness",
    "syncope",
    "overdose",
    "drug overdose",
    "poisoning",
    "toxic ingestion",
    "severe bleeding",
    "hemorrhage",
    "internal bleeding",
    "major trauma",
    "head injury",
    "traumatic brain injury",
    "gunshot wound",
    "stab wound",
    "assault",
    "violence",
    "suicidal ideation",
    "suicide attempt",
    "homicidal ideation",
    "sepsis",
    "septic shock",
    "shock",
    "high fever",
    "burn",
    "major burn",
    "spinal injury",
    "paralysis",
    "severe dehydration",
    "active labor",
    "ectopic pregnancy",
    "vaginal bleeding",
    "postpartum bleeding",
    "pulmonary embolism",
    "aortic dissection",
    "arrhythmia",
    "ventricular tachycardia",
    "ventricular fibrillation",
    "bradycardia",
    "tachycardia",
    "hypoxia",
    "oxygen saturation low",
]

NEGATION_PHRASES = [
    "negative for",
    "no evidence of",
    "without",
    "denies",
    "denied",
    "free of",
    "absence of",
    "absent",
    "no",
    "not",
]
NEGATION_TOKENS = {"no", "not", "denies", "denied", "without", "negative", "free", "absence", "absent", "never"}


def normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""

    t = str(text).lower().strip()
    t = t.replace("_", " ")
    t = re.sub(r"[/|]+", " ", t)
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    if t in PLACEHOLDER_VALUES:
        return ""

    for abbr, expanded in ABBREV_MAP.items():
        t = re.sub(rf"\b{re.escape(abbr)}\b", expanded, t)

    t = re.sub(r"\s+", " ", t).strip()
    if t in PLACEHOLDER_VALUES:
        return ""
    return t


def is_negated(text: str, start_idx: int) -> bool:
    left_context = text[max(0, start_idx - 100):start_idx]
    # Check only the current phrase segment to avoid cross-sentence negation leakage.
    segment = re.split(r"[.;,]", left_context)[-1].strip()
    if not segment:
        return False

    for phrase in NEGATION_PHRASES:
        if phrase in segment:
            return True

    tokens = re.findall(r"[a-z0-9']+", segment)
    if any(tok in NEGATION_TOKENS for tok in tokens[-6:]):
        return True

    return False


def build_keyword_pattern(keyword: str) -> re.Pattern:
    return re.compile(rf"(?<!\w){re.escape(keyword)}(?!\w)")


def make_flag_column_name(keyword: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", keyword.lower()).strip("_")


keyword_patterns = {kw: build_keyword_pattern(kw) for kw in EMERGENCY_KEYWORDS}
keyword_columns = {kw: make_flag_column_name(kw) for kw in EMERGENCY_KEYWORDS}

if len(set(keyword_columns.values())) != len(keyword_columns):
    raise ValueError("Duplicate column names detected from keyword list.")

for col in TEXT_COLS:
    df[f"__proc_{col}"] = df[col].map(normalize_text)

combined_text = (
    df[[f"__proc_{col}" for col in TEXT_COLS]]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(f"Rows loaded: {len(df):,}")
print(f"Emergency keywords configured: {len(EMERGENCY_KEYWORDS)}")
print(f"Final output target: {FINAL_OUTPUT_PATH.resolve()}")
print("Preview of processed text:")
display(combined_text.head(3))

Rows loaded: 58,124
Emergency keywords configured: 66
Final output target: E:\kaggle\traigegeist\working_data\nhamcs_emergency_keyword_flags_matched_only.csv
Preview of processed text:


0                                fever throat soreness
1    injury other and unspecified of foo foot and t...
2                                          fever cough
dtype: str

In [13]:
def keyword_present_non_negated(text: str, pattern: re.Pattern) -> int:
    if not text:
        return 0

    for match in pattern.finditer(text):
        if not is_negated(text, match.start()):
            return 1
    return 0

text_values = combined_text.tolist()
flag_data = {}

for keyword, pattern in keyword_patterns.items():
    col_name = keyword_columns[keyword]
    flag_data[col_name] = np.fromiter(
        (keyword_present_non_negated(text, pattern) for text in text_values),
        dtype=np.int8,
        count=len(text_values),
    )

flags_df = pd.DataFrame(flag_data, index=df.index)
flags_df.insert(0, "row_index", flags_df.index.astype(np.int64))

binary_ok = flags_df.drop(columns=["row_index"]).isin([0, 1]).all().all()
prevalence = flags_df.drop(columns=["row_index"]).sum().sort_values(ascending=False)
prevalence_pct = (prevalence / len(flags_df) * 100).round(3)

summary_df = pd.DataFrame(
    {
        "matches": prevalence.astype(int),
        "percent": prevalence_pct,
    }
)

# Keep only requested matched columns and remove overdose/poisoning.
requested_columns = [
    "chest_pain",
    "shortness_of_breath",
    "syncope",
    "assault",
    "vaginal_bleeding",
    "violence",
    "burn",
    "head_injury",
    "overdose",
    "suicide_attempt",
    "cardiac_arrest",
    "gunshot_wound",
    "throat_swelling",
    "poisoning",
    "paralysis",
    "sepsis",
]
explicit_remove = {"overdose", "poisoning"}

requested_columns = [c for c in requested_columns if c not in explicit_remove]
requested_available = [c for c in requested_columns if c in summary_df.index]
requested_matched = [c for c in requested_available if summary_df.loc[c, "matches"] > 0]

final_df = flags_df[["row_index"] + requested_matched].copy()
final_df.to_csv(FINAL_OUTPUT_PATH, index=False)

final_summary_df = summary_df.loc[requested_matched].copy()

print(f"Binary-only check passed: {binary_ok}")
print(f"Final saved keyword flags: {FINAL_OUTPUT_PATH.resolve()}")
print(f"Final output shape (row_index + selected matched columns): {final_df.shape}")
print(f"Final selected columns: {requested_matched}")

display(final_summary_df)
display(final_df.head(5))

Binary-only check passed: True
Final saved keyword flags: E:\kaggle\traigegeist\working_data\nhamcs_emergency_keyword_flags_matched_only.csv
Final output shape (row_index + selected matched columns): (58124, 15)
Final selected columns: ['chest_pain', 'shortness_of_breath', 'syncope', 'assault', 'vaginal_bleeding', 'violence', 'burn', 'head_injury', 'suicide_attempt', 'cardiac_arrest', 'gunshot_wound', 'throat_swelling', 'paralysis', 'sepsis']


,matches,percent
chest_pain,4083,7.025
shortness_of_breath,4057,6.980
syncope,765,1.316
assault,655,1.127
vaginal_bleeding,411,0.707
violence,370,0.637
burn,193,0.332
head_injury,103,0.177
suicide_attempt,87,0.150
cardiac_arrest,74,0.127


,row_index,chest_pain,shortness_of_breath,syncope,assault,vaginal_bleeding,violence,burn,head_injury,suicide_attempt,cardiac_arrest,gunshot_wound,throat_swelling,paralysis,sepsis
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,1,0,0,0,1,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
display(final_summary_df)

,matches,percent
chest_pain,4083,7.025
shortness_of_breath,4057,6.980
syncope,765,1.316
assault,655,1.127
vaginal_bleeding,411,0.707
violence,370,0.637
burn,193,0.332
head_injury,103,0.177
suicide_attempt,87,0.150
cardiac_arrest,74,0.127
